**Extraindo PDF**

In [1]:
!pip install transformers datasets accelerate sentencepiece PyMuPDF torch huggingface_hub python-dotenv


  Using cached transformers-5.10.2-py3-none-any.whl.metadata (33 kB)
  Using cached datasets-5.0.0-py3-none-any.whl.metadata (23 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached sentencepiece-0.2.1-cp314-cp314-win_amd64.whl.metadata (10 kB)
  Using cached pymupdf-1.27.2.3-cp310-abi3-win_amd64.whl.metadata (24 kB)
  Using cached torch-2.12.0-cp314-cp314-win_amd64.whl.metadata (31 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-win_amd64.whl.metadata (2.4 kB)
  Using cached regex-2026.5.9-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached click-8.4.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached anyio-4.13.0-py3-none

In [2]:
!pip install ipywidgets bitsandbytes

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl.metadata (10 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
Using cached bitsandbytes-0.49.2-py3-none-win_amd64.whl (55.4 MB)

   ---------------------------------------- 0/4 [widgetsnbextension]
   ---------- ----------------------------- 1/4 [jupyterlab_widgets]
   -------------------- ------------------- 2/4 [ipywidgets]
   -------------------- ------------------- 2/4 [ipywidgets]
   -------------------- ------------------- 2/4 [ipywidgets]
   -------------------- ------------------- 2/4 [ipywidgets]
   -------------------- ------------------- 2/4 [ipywidgets]
   --------

In [3]:
import json
import re
# import pdfplumber
import fitz  # PyMuPDF
import re
import torch
from pathlib import Path
# from PyPDF2 import PdfReader
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

# Sem login() — token vai direto nos métodos abaixo
print("Token carregado:", "✅" if HF_TOKEN else "❌ Token não encontrado no .env")

Token carregado: ✅


**Extração do PDF**

In [5]:
def extrair_texto_pdf(caminho_pdf: str, skip_pages: int = 0):
    """
    Retorna lista de (numero_pagina, texto) para todo o PDF.
    """
    paginas = []
    doc = fitz.open(caminho_pdf)
    for i in range(skip_pages, len(doc)):
        page = doc[i]
        texto = page.get_text() or ""
        paginas.append((i + 1, texto))
    doc.close()
    return paginas

pdf_path = "20260018701.pdf"
paginas = extrair_texto_pdf(pdf_path, skip_pages=4)

print(f"Total de páginas extraídas: {len(paginas)}")
print("\n--- EXEMPLO ---")
print(paginas[4][1][:300])  # pág. 37, onde começa o Alecrim

Total de páginas extraídas: 152

--- EXEMPLO ---
elas defendem umas às outras, fazem uma guerra química pra
ajudar a vizinha, elas conversam umas com as outras, tem uma
comunicação entre elas. As plantas estão em comunidade.
A medicina da simplicidade | 9



In [7]:
# Filtra para páginas até a 117 (inclusive), o restante é apenas referências e anexos
paginas = [(n, t) for n, t in paginas if n <= 117]

In [8]:
# Detecta o título de início de ficha nas páginas
# Exemplo: "Alecrim -RosmarinusOfficinalis|37"
PADRAO_TITULO = re.compile(
    r"^([A-ZÀ-Ú][^\|]+?)\s*-\s*([A-Z][a-z]+\s+[A-Za-z]+[^\|]*)\|\s*(\d+)$",
    re.MULTILINE
)
for num_pag, texto in paginas:
    for linha in texto.split("\n"):
        m = PADRAO_TITULO.match(linha.strip())
        if m:
            print(f"Pág {num_pag}: {repr(linha.strip())}")

Pág 37: 'Alecrim - Rosmarinus Officinalis | 37'
Pág 39: 'Alfavaca Anisada - Ocimum selloi | 39'
Pág 41: 'Alfavaca Cravo - Ocimum gratissimum | 41'
Pág 43: 'Manjericão - Ocimum americanum. | 43'
Pág 47: 'Arnica - Sphagneticola trilobata | 47'
Pág 49: 'Arnica - Solidago chilensis | 49'
Pág 51: 'Arnica - Calea uniflora | 51'
Pág 53: 'Aveia - Avena sativa | 53'
Pág 55: 'Babosa - Aloe vera | 55'
Pág 57: 'Boldo - Plectranthus barbatus | 57'
Pág 59: 'Boldinho - Plectranthus ornatus | 59'
Pág 61: 'Malvariço - Plectranthus amboinicus | 61'
Pág 63: 'Boldo-baiano - Gymnanthemum amygdalinum | 63'
Pág 65: 'Calêndula - Calendula officinalis | 65'
Pág 67: 'Camomila - Matricaria recutita | 67'
Pág 69: 'Chapéu-de-couro - Echinodorus grandiflorus | 69'
Pág 71: 'Erva-cidreira - Melissa officinalis | 71'
Pág 73: 'Cidró - Aloysia triphylla | 73'
Pág 75: 'Capim-cidreira - Cymbopogom citratus | 75'
Pág 77: 'Salva-da-gripe - Lippia alba | 77'
Pág 79: 'Cúrcuma - Curcuma longa | 79'
Pág 81: 'Erva-baleeira - Var

**Divisão por chunks e nomes de plantas**

In [9]:
def dividir_por_plantas(paginas: list) -> list:
    """
    Agrupa as páginas em fichas completas de cada planta.
    Retorna lista de dicts:
      { "planta": "Camomila", "pagina_inicio": 67, "texto": "..." }
    """
    fichas = []
    ficha_atual = None

    for num_pag, texto in paginas:
        for linha in texto.split("\n"):
            m = PADRAO_TITULO.match(linha.strip())
            if m:
                if ficha_atual is not None:
                    fichas.append(ficha_atual)
                nome = m.group(1).strip()
                ficha_atual = {
                    "planta": nome,
                    "pagina_inicio": num_pag,
                    "texto": texto,
                }
                break
        else:
            if ficha_atual is not None:
                ficha_atual["texto"] += "\n" + texto

    if ficha_atual is not None:
        fichas.append(ficha_atual)

    return fichas

In [10]:
fichas = dividir_por_plantas(paginas)

print(f"Fichas encontradas: {len(fichas)}\n")
for f in fichas:
    print(f"  • {f['planta']:30s} (pág. {f['pagina_inicio']})")

Fichas encontradas: 39

  • Alecrim                        (pág. 37)
  • Alfavaca Anisada               (pág. 39)
  • Alfavaca Cravo                 (pág. 41)
  • Manjericão                     (pág. 43)
  • Arnica                         (pág. 47)
  • Arnica                         (pág. 49)
  • Arnica                         (pág. 51)
  • Aveia                          (pág. 53)
  • Babosa                         (pág. 55)
  • Boldo                          (pág. 57)
  • Boldinho                       (pág. 59)
  • Malvariço                      (pág. 61)
  • Boldo-baiano                   (pág. 63)
  • Calêndula                      (pág. 65)
  • Camomila                       (pág. 67)
  • Chapéu-de-couro                (pág. 69)
  • Erva-cidreira                  (pág. 71)
  • Cidró                          (pág. 73)
  • Capim-cidreira                 (pág. 75)
  • Salva-da-gripe                 (pág. 77)
  • Cúrcuma                        (pág. 79)
  • Erva-baleeira              

In [ ]:
LINHAS_RUIDO = re.compile(
    r"^(\d+|Colaboradores|Programa de Pós|UFSC\."
    r"|PARTES USADAS|CARACTERÍSTICAS BOTÂNICAS"
    r"|Sistema[s]?:)",
    re.IGNORECASE,
)

def chunk_por_planta(fichas: list, max_chars: int = 600, overlap=50) -> list:
    """
    Divide cada ficha em chunks com o prefixo "PLANTA: <nome>"
    garantindo que o modelo sempre saiba qual planta está processando.
    """
    chunks = []

    for ficha in fichas:
        planta = ficha["planta"]
        texto = ficha["texto"]

        linhas = [l.strip() for l in texto.split("\n") if l.strip()]
        paragrafos = [l for l in linhas if not LINHAS_RUIDO.match(l)]

        chunk_atual = ""
        for paragrafo in paragrafos:
            if len(chunk_atual) + len(paragrafo) + 1 <= max_chars:
                chunk_atual += ("\n" if chunk_atual else "") + paragrafo
            else:
                if chunk_atual:
                    chunks.append({
                        "planta": planta,
                        "chunk": f"PLANTA: {planta}\n\n{chunk_atual}",
                    })
                if len(paragrafo) > max_chars:
                    palavras = paragrafo.split()
                    sub = ""
                    for palavra in palavras:
                        if len(sub) + len(palavra) + 1 <= max_chars:
                            sub += (" " if sub else "") + palavra
                        else:
                            if sub:
                                chunks.append({
                                    "planta": planta,
                                    "chunk": f"PLANTA: {planta}\n\n{sub}",
                                })
                            sub = palavra
                    chunk_atual = sub
                else:
                    chunk_atual = paragrafo

        if chunk_atual:
            chunks.append({
                "planta": planta,
                "chunk": f"PLANTA: {planta}\n\n{chunk_atual}",
            })

    return chunks

In [15]:
# Teste

chunks = chunk_por_planta(fichas, max_chars=600)

print(f"Total de chunks: {len(chunks)}")
print(f"\n--- EXEMPLO (primeiro chunk do Alecrim) ---")
print(chunks[0]["chunk"])

Total de chunks: 229

--- EXEMPLO (primeiro chunk do Alecrim) ---
PLANTA: Alecrim

Arnicas: Amanda Ellen de Athayde: Programa de Pós gradu-
ação em Ciências Farmacêuticas da UFSC.
Cúrcuma: Ana Júlia Lobo Feijó: Programa de Pós-graduação
em Saúde coletiva da UFSC.
Erva-de-santa-maria: Larissa Frankenberger: Programa de
Pós graduação em Ciências Farmacêuticas da UFSC.
Demais espécies: Michael Anderson da Luz Lopes: Grad-
uando do curso de Farmácia da UFSC. Rafaela de Jesus
Souza: Farmacêutica formada pela UFSC
Plantas
Rosmarinus officinalis L., Lamiaceae
Alecrim
planta herbácea, perene, aromática, amplamente cultivada em


In [16]:
print("="*80)
print(f"Total de chunks: {len(chunks)}")
print("="*80)

for i, chunk_dict in enumerate(chunks[:10]):
    nome_planta = chunk_dict["planta"]
    preview = chunk_dict["chunk"][:200].replace('\n', ' ')

    print(f"\n📌 CHUNK #{i+1}")
    print(f"   🌿 Planta: {nome_planta}")
    print(f"   📝 Início: {preview}...")
    print(f"   📏 Tamanho: {len(chunk_dict['chunk'])} caracteres")
    print("-"*80)

Total de chunks: 229

📌 CHUNK #1
   🌿 Planta: Alecrim
   📝 Início: PLANTA: Alecrim  Arnicas: Amanda Ellen de Athayde: Programa de Pós gradu- ação em Ciências Farmacêuticas da UFSC. Cúrcuma: Ana Júlia Lobo Feijó: Programa de Pós-graduação em Saúde coletiva da UFSC. Er...
   📏 Tamanho: 560 caracteres
--------------------------------------------------------------------------------

📌 CHUNK #2
   🌿 Planta: Alecrim
   📝 Início: PLANTA: Alecrim  hortas domésticas em Florianópolis. A espécie pode ser propa- gada por estacas produzidas a partir dos seus ramos e o seu cul- tivo deve ser preferencialmente realizado em lugares com...
   📏 Tamanho: 569 caracteres
--------------------------------------------------------------------------------

📌 CHUNK #3
   🌿 Planta: Alecrim
   📝 Início: PLANTA: Alecrim  blemas hepáticos, problemas renais, distúrbios estomacais, dores de cabeça, bronquites e asma. Usado externamente para lavagem de feridas, afecções do couro cabeludo e em banhos para ...
   📏 Tama

**Carregando modelo**

In [19]:
# Alternativa para GPU com pouca VRAM
from transformers import BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"
#bnb_config = BitsAndBytesConfig(load_in_4bit=True)

tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=HF_TOKEN
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=HF_TOKEN,
    #quantization_config=bnb_config,  # substitui o torch_dtype
    device_map="auto"
)


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

**Funções auxiliares**

In [25]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def is_bad_chunk(text, min_len=80):
    text = text.strip()
    if len(text) < min_len:
        return True
    words = text.split()
    if len(set(words)) < 10:
        return True
    strange_ratio = sum(1 for c in text if not c.isalnum() and c not in " .,;:-()[]{}") / max(len(text), 1) #
    if strange_ratio > 0.3:
        return True
    return False

def run_model_llama(model, tokenizer, chunk_dict):  # ← chunk_dict em vez de chunk
    messages = build_messages(chunk_dict)            # ← passa o dict

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    terminators = [
        tokenizer.eos_token_id,
        tokenizer.convert_tokens_to_ids("<|eot_id|>")
    ]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            eos_token_id=terminators
        )

    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][prompt_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


In [26]:
EXPRESSOES_PROIBIDAS = [
    "a planta mencionada", "a planta descrita", "a espécie citada",
    "o vegetal apresentado", "a erva em questão", "a espécie mencionada",
    "a planta em questão", "o texto menciona", "conforme o texto",
]

def extract_triple(generated_text: str, nome_planta: str):
    texto = generated_text.strip().upper()
    if texto.startswith("PULAR") or texto == "PULAR":
        return None

    m_perg = re.search(
        r"PERGUNTA\s*:\s*(.+?)(?=RESPOSTA\s*:|$)", generated_text, re.DOTALL | re.IGNORECASE
    )
    m_resp = re.search(r"RESPOSTA\s*:\s*(.+)", generated_text, re.DOTALL | re.IGNORECASE)

    if not m_perg or not m_resp:
        return None

    pergunta = m_perg.group(1).strip()
    resposta = m_resp.group(1).strip()

    if len(pergunta) < 15 or len(resposta) < 30:
        return None

    texto_completo = (pergunta + " " + resposta).lower()

    for expr in EXPRESSOES_PROIBIDAS:
        if expr.lower() in texto_completo:
            return None

    # Exige que pelo menos uma palavra do nome apareça no par
    palavras_nome = nome_planta.lower().split()
    if not any(p in texto_completo for p in palavras_nome):
        return None

    return {
        "Instruction": pergunta,
        "Output": resposta,
    }

In [27]:
def build_messages(chunk_dict: dict) -> list:
    nome_planta = chunk_dict["planta"]
    # Remove prefixo duplicado caso já exista no chunk
    texto_base = chunk_dict["chunk"]
    if texto_base.startswith(f"PLANTA: {nome_planta}"):
        texto_base = texto_base[len(f"PLANTA: {nome_planta}"):].lstrip()

    chunk_texto = f"PLANTA: {nome_planta}\n\n{texto_base}"

    system_prompt = f"""Você é um especialista em plantas medicinais e curador de datasets para treinamento de modelos RAG.

A planta desta ficha é: {nome_planta}

Sua tarefa é gerar EXATAMENTE UMA pergunta e UMA resposta completa e detalhada baseadas SOMENTE no texto fornecido.

━━━ QUANDO RESPONDER "PULAR" ━━━
Responda SOMENTE a palavra PULAR se o texto for claramente inutilizável:
- Referência bibliográfica (autor + ano + revista/DOI)
- Somente o nome científico isolado, sem nenhuma informação sobre a planta
- Texto com menos de 10 palavras que não forma uma informação completa

Em TODOS os outros casos, gere o par — mesmo que o conteúdo seja simples.

━━━ QUANDO GERAR O PAR ━━━
Gere uma pergunta e resposta SE o texto tiver informação útil sobre:
- usos populares e indicações terapêuticas
- contraindicações e efeitos adversos
- modo de preparo, dosagem e formas de uso
- propagação, cultivo e partes utilizadas
- composição química e características botânicas
- evidências científicas e cuidados de uso

━━━ REGRAS OBRIGATÓRIAS ━━━
1. Use APENAS informações presentes no texto. Nunca invente.
2. O nome da planta é "{nome_planta}". Use EXATAMENTE este nome.
3. PROIBIDO usar: "a planta mencionada", "a espécie citada", "a planta descrita", "o vegetal apresentado", "a erva em questão", "a planta em questão".
4. A pergunta deve ser compreensível sem depender do texto.
5. A resposta deve ter no mínimo 15 palavras e ser completa.
6. Prefira perguntas sobre USO, PREPARO, CONTRAINDICAÇÕES ou EFEITOS — não sobre família botânica.
7. Preserve nomes científicos e famílias botânicas exatamente como aparecem no texto.
8. Não faça inferências nem use conhecimento externo.

━━━ FORMATO DE SAÍDA ━━━
Se for pular:
PULAR

Se for gerar:
PERGUNTA: ...
RESPOSTA: ..."""

    user_prompt = f"""━━━ EXEMPLO 1 — chunk útil, gerar par ━━━

PERGUNTA: Quais são os usos populares e formas de preparo da Camomila?
RESPOSTA: A Camomila é utilizada na forma de infusão dos capítulos florais para
cólicas menstruais, cólicas pós-parto e distúrbios gastrointestinais, atuando como
antiespasmódico e sedativo leve. A infusão é preparada com 1 colher de sobremesa
dos capítulos florais para 1 xícara de água fervente, abafando por 10 minutos.
Externamente, é empregada em cólicas infantis, doenças da pele, candidíase e
hemorróidas na forma de banhos de assento.

━━━ EXEMPLO 2 — chunk de referência bibliográfica, PULAR ━━━

TEXTO:
PLANTA: Tansagem

1. VANACLOCHA, B.; CAÑIGUERAL, S. Fitoterapia: vademécum de prescripción. 4. ed. Barcelona: Masson, 2003.
2. BLUMENTHAL, M. et al. The complete German Commission E monographs. Austin: American Botanical Council, 1998.

PULAR

━━━ EXEMPLO 3 — chunk com contraindicação, gerar par ━━━

TEXTO:
PLANTA: Boldo

CUIDADOS NO USO DESTA ESPÉCIE: Não deve ser utilizado durante a gestação e lactação. O uso prolongado pode causar danos hepáticos. Evitar uso concomitante com anticoagulantes.

PERGUNTA: Quais são os cuidados e contraindicações no uso do Boldo?
RESPOSTA: O Boldo não deve ser utilizado durante a gestação e lactação. O uso prolongado pode causar danos hepáticos e deve-se evitar o uso concomitante com anticoagulantes.

━━━ EXEMPLO 4 — chunk fragmentado sem valor, PULAR ━━━

TEXTO:
PLANTA: Hortelã

Lamiaceae
Mentha spp.

PULAR

━━━ AGORA PROCESSE ESTE TEXTO ━━━

{chunk_texto}"""

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

In [28]:
clean_chunks = [
    {"planta": c["planta"], "chunk": clean_text(c["chunk"])}
    for c in chunks
    if not is_bad_chunk(c["chunk"])
]
print(f"Chunks válidos: {len(clean_chunks)}")

Chunks válidos: 224


**Gerando dataset**

In [29]:
validos = 0
triplas_causal = []

amostra_chunks = clean_chunks[:5]

print("=== INICIANDO AMOSTRA DE TESTE COM LLAMA 3.2 ===")

for i, chunk_dict in enumerate(amostra_chunks):
    planta = chunk_dict["planta"]

    try:
        # Verifica se o chunk tem conteúdo útil
        if is_bad_chunk(chunk_dict["chunk"]):
            print(f"  [{i+1}/{len(amostra_chunks)}] ⏭  {planta}: chunk ruim, pulado")
            continue

        messages = build_messages(chunk_dict)
        resposta_modelo = run_model_llama(model, tokenizer, chunk_dict)
        par = extract_triple(resposta_modelo, planta)  # ← agora passa o nome também

        if par:
            triplas_causal.append(par)
            validos += 1
            print(f"  [{i+1}/{len(amostra_chunks)}] ✅ {planta}: {par['Instruction'][:60]}...")
        else:
            print(f"  [{i+1}/{len(amostra_chunks)}] ⏭  {planta}: par inválido/pulado")

    except Exception as e:
        print(f"  [{i+1}/{len(amostra_chunks)}] ❌ {planta}: erro — {e}")

print(f"\nPares válidos gerados: {validos}/{len(amostra_chunks)}")

[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.


=== INICIANDO AMOSTRA DE TESTE COM LLAMA 3.2 ===
  [1/5] ❌ Alecrim: erro — Tensor.item() cannot be called on meta tensors
  [2/5] ❌ Alecrim: erro — Tensor.item() cannot be called on meta tensors
  [3/5] ❌ Alecrim: erro — Tensor.item() cannot be called on meta tensors
  [4/5] ❌ Alecrim: erro — Tensor.item() cannot be called on meta tensors
  [5/5] ❌ Alecrim: erro — Tensor.item() cannot be called on meta tensors

Pares válidos gerados: 0/5


In [30]:
triplas_causal = []


print(f"=== INICIANDO GERAÇÃO DO DATASET OFICIAL COM 600 CHARS ({len(clean_chunks)} chunks) ===")

for i, chunk_dict in enumerate(clean_chunks):  # ← chunk_dict em vez de chunk

    chunk_limpo = clean_text(chunk_dict["chunk"])  # ← acessa o texto do dict

    if is_bad_chunk(chunk_limpo, min_len=80):
        print(f"⏭️ Chunk {i}: Ignorado por critérios de qualidade mínimos.")
        continue

    try:
        generated = run_model_llama(model, tokenizer, chunk_dict)  # ← passa o dict

        triple = extract_triple(generated, chunk_dict["planta"])  # ← passa o nome

        if triple:
            triplas_causal.append(triple)

        if i % 40 == 0 and i > 0:
            print(f"Processados {i}/{len(clean_chunks)} chunks... Triplas válidas até agora: {len(triplas_causal)}")

    except Exception as e:
        print(f"⚠️ Erro ao processar o chunk índice {i}: {e}")

print("\n" + "="*50)
print(f"🎉 FIM DO PROCESSAMENTO DO DATASET OFICIAL!")
print(f"Total de chunks processados: {len(clean_chunks)}")
print(f"Total de pares válidos gerados com sucesso: {len(triplas_causal)}")
print(f"Taxa de aproveitamento: {100*len(triplas_causal)/len(clean_chunks):.1f}%")
print("="*50)

[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_to

=== INICIANDO GERAÇÃO DO DATASET OFICIAL COM 600 CHARS (224 chunks) ===
⚠️ Erro ao processar o chunk índice 0: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 1: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 2: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 3: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 4: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 5: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 6: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 7: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 8: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 9: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 10: Tensor.item() cannot be call

[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_to

⚠️ Erro ao processar o chunk índice 15: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 16: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 17: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 18: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 19: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 20: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 21: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 22: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 23: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 24: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 25: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 26: Ten

[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_to

...

⚠️ Erro ao processar o chunk índice 166: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 167: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 168: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 169: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 170: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 171: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 172: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 173: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 174: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 175: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 176: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índ

[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_to

⚠️ Erro ao processar o chunk índice 183: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 184: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 185: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 186: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 187: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 188: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 189: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 190: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 191: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 192: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 193: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índ

[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:tensor(..., device='meta', size=(), dtype=torch.int64) for open-end generation.
[transformers] Setting `pad_token_id` to `eos_to

⚠️ Erro ao processar o chunk índice 203: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 204: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 205: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 206: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 207: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 208: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 209: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 210: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 211: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 212: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índice 213: Tensor.item() cannot be called on meta tensors
⚠️ Erro ao processar o chunk índ

In [31]:
def clean_triple(triple):
    # 1. Remove espaços em branco das pontas de todas as chaves existentes
    for key in triple:
        triple[key] = triple[key].strip()

    # 2. Mantém a sua excelente regra de tamanho mínimo (com as chaves corrigidas)
    if len(triple["Output"]) < 20 or len(triple["Instruction"]) < 5:
        return None

    return triple

def clean_list(triple):
    # Filtra triplas inválidas após a limpeza
    return [t for t in (clean_triple(x) for x in triple) if t is not None]


# Executa a limpeza na sua lista final
triplas_causal = clean_list(triplas_causal)

print(f"Triplas causal após limpeza final: {len(triplas_causal)}")

Triplas causal após limpeza final: 0


In [259]:
def mostrar_exemplo(triplas, titulo, n=3):
    print(f"\n=== {titulo} ===")
    for i, t in enumerate(triplas[:n], 1):
        print(f"\n--- Exemplo {i} ---")
        print(json.dumps(t, indent=2, ensure_ascii=False))

mostrar_exemplo(triplas_causal, "meta-llama/Llama-3.2-3B-Instruct")


=== meta-llama/Llama-3.2-3B-Instruct ===

--- Exemplo 1 ---
{
  "Instruction": "Qual é a definição de sinergismo no contexto das plantas medicinais?",
  "Output": "O sinergismo é uma das coisas mais importantes no mundo das plantas, referindo-se à interação de todos os princípios ativos isolados, concentrados, sintetizados, que fazem o remédio, trabalhando juntos, e que é a energia da planta que não se pode descrever sem a presença de todos os princípios ativos."
}

--- Exemplo 2 ---
{
  "Instruction": "O que é considerado \"sinergismo\" na plantas e como isso se relaciona com a isolamento de princípios ativos?",
  "Output": "O sinergismo na planta se refere à energia de cura que não pode ser isolada apenas com princípios ativos, e que requer uma compreensão mais esotérica e intuitiva, que não é tão comum em estudos científicos."
}

--- Exemplo 3 ---
{
  "Instruction": "Como as plantas interagem entre as diferentes partes de suas estruturas e como elas lidam com os princípios ativos e

In [ ]:
def salvar_jsonl(triplas, nome_arquivo):
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        for t in triplas:
            f.write(json.dumps(t, ensure_ascii=False) + "\n")
    print(f"Dataset salvo em {nome_arquivo}")

salvar_jsonl(triplas_causal, "dataset_final_divisoes.jsonl")

Dataset salvo em dataset_teste_final.jsonl
